#### Aufgabe 1: Verbindung & Grundoperationen (String)

In [1]:
import redis
c = redis.Redis(host='localhost', port=6379, db=0)
c.set("username", "alice")
print(c.get("username"))
print(c.exists("username"))

b'alice'
1


#### Aufgabe2: Ablaufzeit (TTL)

In [4]:
c.setex("session_token", 10, "xyz123")


True

#### Aufgabe 3: Zähler (String-Integer)

In [5]:
c.set("counter", 1)
c.incrby("counter", 1)
c.get("counter")

b'2'

#### Aufgabe 4: Hash – Benutzerprofil

In [4]:
c.hset("user:1001", "name", "Alice")
c.hset("user:1001", "age", "30")
c.hset("user:1001", "email", "alice@example.com")
print(c.hgetall("user:1001"))
c.hset("user:1001", "age", "31")
print(c.hgetall("user:1001"))

{b'name': b'Alice', b'age': b'30', b'email': b'alice@example.com'}
{b'name': b'Alice', b'age': b'31', b'email': b'alice@example.com'}


#### Aufgabe 5: Liste – Aufgaben-Queue

In [5]:
c.lpush("tasks", "send_mail", "generate_report", "backup_db")
print(c.rpop("tasks"))
print(c.rpop("tasks"))
print(c.rpop("tasks"))
print(c.rpop("tasks"))

b'send_mail'
b'generate_report'
b'backup_db'
None


#### Aufgabe 6: Set – Eindeutige Besucher

In [4]:
c.sadd("unique_visitors", "192.168.0.1", "192.168.0.2", "192.168.0.1")
print(c.scard("unique_visitors"))
print(c.smembers("unique_visitors"))

2
{b'192.168.0.1', b'192.168.0.2'}


#### Aufgabe 7: Sorted Set – Rangliste

In [8]:
c.zadd("leaderboard", {"Alice": 50, "Bob": 70, "Calol": 60})
c.zincrby("leaderboard", 15, "Alice")
print(c.zrange("leaderboard", 0, -1, withscores=True))

[(b'Calol', 60.0), (b'Alice', 65.0), (b'Bob', 70.0)]


#### Aufgabe 8: Transaktion

In [9]:
with c.pipeline() as pipe:  
	pipe.set('coins', 1)  
	pipe.incrby("coins", 10) 
	pipe.execute()
print(c.get("coins"))

b'11'


#### Aufgabe 9: Pipeline – Performance

In [13]:
for i in range(0, 1000):
    key = "classicItem:" + str(i)
    c.set(key, i)

In [14]:
with c.pipeline() as pipe:
    for i in range(0, 1000):
        key = "pipeItem:" + str(i)
        pipe.set(key, i)
    pipe.execute()

#### Aufgabe 10: Pub/Sub – Nachrichtenübertragung

In [15]:
import threading

def publish_message():  
	while True:  
		message = input("Enter message to publish: ")  
		c.publish('my_channel', message) 
		 
def subscribe_to_messages():  
	pubsub = c.pubsub()  
	pubsub.subscribe('my_channel')  
	for message in pubsub.listen():  
		if message['type'] == 'message':  
			print(f"Received: {message['data'].decode('utf-8')}") 
			 
# Start des publisher and subscriber in eigenen Threads  
threading.Thread(target=publish_message).start()  
threading.Thread(target=subscribe_to_messages).start()

Received: a
Enter message to publish: 

#### Aufgabe 11: Expiring Cache Layer

In [6]:
def getOrCache(key, function, ttl):
    firstTry = c.get(key)
    if firstTry != None:
        print("Found")
        return firstTry
    else:
        written = c.set(key, function(), ttl)
        print("Written")
        return written

import random
def randValue():
    return random.randint(0, 10)
print(getOrCache("a", randValue, 5))

Found
b'7'


#### Expertenaufgabe 12: Stream – Ereignisprotokoll

1770565417960-0 {'type': 'login', 'user': 'max'}
1770565417959-0 {'type': 'purchase', 'user': 'bob'}


#### Expertenaufgabe 13:Mini-Rate-Limiter

In [18]:
print(c.get("a"))

None


#### Expertenaufgabe 14:Lua-Script – Atomare Operation

OK
